# 📈 Week 4 — Sales Forecasting with LSTM
**Smart E-Commerce Analytics Platform**

**Dataset:** Olist Brazilian E-Commerce → `orders_clean.csv` (daily revenue aggregated)

**This notebook covers:**
1. Aggregate Olist orders into daily revenue time series
2. Exploratory time series analysis
3. Prepare sequences for LSTM (sliding window)
4. Build & train LSTM model
5. Evaluate: MAE, RMSE, MAPE
6. 30-day future forecast

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.titlesize'] = 14
print('Libraries loaded ✅')

## 1. Build Daily Revenue Time Series

In [ ]:
df = pd.read_csv('../data/orders_clean.csv', parse_dates=['transaction_date'])
print(f'Orders loaded: {len(df):,}')

# Aggregate to daily revenue
daily = (df.groupby(df['transaction_date'].dt.date)['total_amount']
         .sum().reset_index())
daily.columns = ['date', 'revenue']
daily['date'] = pd.to_datetime(daily['date'])

# Fill missing dates with 0
full_range = pd.date_range(daily['date'].min(), daily['date'].max(), freq='D')
daily = daily.set_index('date').reindex(full_range, fill_value=0).reset_index()
daily.columns = ['date', 'revenue']

# 7-day moving average
daily['revenue_smooth'] = daily['revenue'].rolling(7, min_periods=1).mean()

print(f'Daily series: {len(daily)} days')
print(f'Date range: {daily["date"].min().date()} → {daily["date"].max().date()}')
print(f'Total revenue: R${daily["revenue"].sum():,.0f}')
print(f'Avg daily revenue: R${daily["revenue"].mean():,.0f}')
display(daily.head())

## 2. Time Series Exploration

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Daily revenue
axes[0].plot(daily['date'], daily['revenue'], alpha=0.3, color='steelblue', linewidth=0.8)
axes[0].plot(daily['date'], daily['revenue_smooth'], color='steelblue', linewidth=2,
             label='7-Day Moving Average')
axes[0].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[0].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
axes[0].set_title('Daily Revenue — Olist 2016–2018')
axes[0].set_ylabel('Revenue (R$)')
axes[0].legend()

# Monthly aggregation
monthly = daily.groupby(daily['date'].dt.to_period('M'))['revenue'].sum().reset_index()
monthly['date'] = monthly['date'].dt.to_timestamp()
axes[1].bar(monthly['date'], monthly['revenue'], width=20, color='steelblue', edgecolor='white')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
axes[1].set_title('Monthly Revenue (R$)')
axes[1].set_ylabel('Revenue (R$)')

plt.tight_layout()
plt.savefig('../report/daily_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Prepare LSTM Sequences

In [ ]:
LOOK_BACK = 30  # Use 30 days to predict next day

series = daily['revenue_smooth'].values.reshape(-1, 1)
scaler = MinMaxScaler()
scaled = scaler.fit_transform(series)

# Create sliding window sequences
X_all, y_all = [], []
for i in range(LOOK_BACK, len(scaled)):
    X_all.append(scaled[i - LOOK_BACK:i, 0])
    y_all.append(scaled[i, 0])

X_all = np.array(X_all).reshape(-1, LOOK_BACK, 1)
y_all = np.array(y_all)

# 80/20 train-test split
split = int(len(X_all) * 0.8)
X_tr, X_te = X_all[:split], X_all[split:]
y_tr, y_te = y_all[:split], y_all[split:]

print(f'Look-back window: {LOOK_BACK} days')
print(f'Total sequences: {len(X_all)}')
print(f'Train sequences: {len(X_tr)}')
print(f'Test sequences:  {len(X_te)}')
print(f'X shape: {X_tr.shape}  (samples, timesteps, features)')

## 4. Build & Train LSTM Model

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

model = Sequential([
    Input(shape=(LOOK_BACK, 1)),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dropout(0.2),
    Dense(16, activation='relu'),
    Dense(1)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

In [ ]:
es = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_tr, y_tr,
    epochs=60,
    batch_size=32,
    validation_data=(X_te, y_te),
    callbacks=[es],
    verbose=1
)

print(f'\nTraining stopped at epoch {len(history.history["loss"])}')

# Plot training loss
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(history.history['loss'],     label='Train Loss')
ax.plot(history.history['val_loss'], label='Val Loss')
ax.set_title('LSTM Training Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.legend()
plt.tight_layout()
plt.savefig('../report/lstm_training_loss.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Evaluate Model

In [ ]:
y_pred_sc = model.predict(X_te, verbose=0)
y_pred    = scaler.inverse_transform(y_pred_sc).flatten()
y_true    = scaler.inverse_transform(y_te.reshape(-1, 1)).flatten()
test_dates = daily['date'].values[LOOK_BACK + split:]

mae  = np.mean(np.abs(y_true - y_pred))
rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100

print(f'Test MAE:  R${mae:,.0f}')
print(f'Test RMSE: R${rmse:,.0f}')
print(f'MAPE:      {mape:.1f}%')

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test_dates, y_true, label='Actual',    color='steelblue', linewidth=2)
ax.plot(test_dates, y_pred, label='Predicted', color='orange',    linewidth=2, linestyle='--')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.xticks(rotation=45)
ax.set_title(f'LSTM — Actual vs Predicted (MAE=R${mae:,.0f}, RMSE=R${rmse:,.0f})')
ax.set_ylabel('Revenue (R$)')
ax.legend()
plt.tight_layout()
plt.savefig('../report/lstm_actual_vs_pred.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. 30-Day Future Forecast

In [ ]:
last_seq  = scaled[-LOOK_BACK:].reshape(1, LOOK_BACK, 1)
fc_scaled = []

for _ in range(30):
    p = model.predict(last_seq, verbose=0)[0, 0]
    fc_scaled.append(p)
    last_seq = np.roll(last_seq, -1, axis=1)
    last_seq[0, -1, 0] = p

forecast  = scaler.inverse_transform(np.array(fc_scaled).reshape(-1, 1)).flatten()
last_date = daily['date'].max()
fc_dates  = pd.date_range(last_date + pd.Timedelta(days=1), periods=30, freq='D')

fig, ax = plt.subplots(figsize=(14, 5))
hist_tail = daily.tail(90)
ax.plot(hist_tail['date'], hist_tail['revenue_smooth'],
        color='steelblue', linewidth=2, label='Historical (7-Day MA)')
ax.plot(fc_dates, forecast, color='red', linewidth=2,
        linestyle='--', marker='o', markersize=4, label='30-Day Forecast')
ax.axvline(x=last_date, color='gray', linestyle=':', linewidth=1.5, label='Forecast Start')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d %b %Y'))
plt.xticks(rotation=45)
ax.set_title('30-Day Sales Forecast (LSTM) — R$')
ax.set_ylabel('Revenue (R$)')
ax.legend()
plt.tight_layout()
plt.savefig('../report/lstm_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

fc_df = pd.DataFrame({'Date': fc_dates.strftime('%Y-%m-%d'), 'Forecast_Revenue_R$': forecast.round(2)})
display(fc_df)
print(f'\nAvg forecasted daily revenue: R${forecast.mean():,.0f}')
print('\n✅ Sales Forecasting Complete!')